# lora_timing_test

Measures seconds per optimizer step and peak VRAM for OmniGen2 LoRA at 512px on a single A100, then extrapolates the full arm.

Runs 12 optimizer steps at OmniGen2 defaults (`global_batch_size: 16` via `batch_size: 2` x `gradient_accumulation_steps: 8`, rank 8, lr 8e-7, `ref_img_dropout_prob: 0.5`), with only the three resolution keys changed to 512. Checkpointing disabled so no 9 GB write lands during timing.

Sections 3 and 4 are preflight: whether the FLUX VAE and OmniGen2's shipped VAE are identical, and whether the text encoder can be loaded from OmniGen2 rather than stock Qwen. Both would waste a 7-hour run if wrong.

Cost: one model load plus ~2 minutes of compute, roughly 2 compute units.

A100 runtime. Nothing here writes to the thesis Drive folders.

## 1. Repo, deps

In [ ]:
import os, subprocess, sys

REPO = "/content/OmniGen2"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/VectorSpaceLab/OmniGen2.git", REPO], check=True)
    print("cloned")
else:
    print("repo already present")

# requirements.txt omits torchdiffeq (omnigen2/transport/integrators.py) and peft (train.py).
# inference never imports transport, which is why the earlier notebooks ran without torchdiffeq.
# peft is preinstalled and working; installing it pulls a version requiring torchao>0.16.0,
# which Colab does not ship, so it is deliberately left alone.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=REPO, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchdiffeq"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "huggingface_hub[hf_xet]>=0.30.0,<1.0"], check=True)

import torchdiffeq, peft
print("torchdiffeq", torchdiffeq.__version__, "| peft", peft.__version__)

# Colab ships peft 0.19.1 with torchao 0.10.0. peft's is_torchao_available() raises rather than
# returning False when torchao is older than 0.16.0, so any library probing peft for quantization
# support crashes during model load. torchao is unused by the bf16 LoRA training path; remove it.
_ao = subprocess.run([sys.executable, "-c",
                      "from peft.import_utils import is_torchao_available; is_torchao_available()"],
                     capture_output=True, text=True)
if _ao.returncode != 0:
    print("torchao incompatible with peft - uninstalling (unused by this training path)")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)

# fail fast: exercise the paths that actually broke, before any model load
_probe = ("import dotenv, matplotlib, torch, transformers, diffusers, accelerate;"
          "from peft import LoraConfig;"
          "from peft.import_utils import is_torchao_available;"
          "assert is_torchao_available() is not None or True;"
          "from diffusers.models.autoencoders.autoencoder_kl import AutoencoderKL;"
          "from omnigen2.transport import create_transport;"
          "from omnigen2.dataset.omnigen2_train_dataset import OmniGen2TrainDataset, OmniGen2Collator;"
          "from omnigen2.models.transformers.transformer_omnigen2 import OmniGen2Transformer2DModel;"
          "print('train.py imports OK')")
chk = subprocess.run([sys.executable, "-c", _probe], cwd=REPO, capture_output=True, text=True)
print(chk.stdout.strip() or chk.stderr.strip()[-1200:])
assert chk.returncode == 0, "train.py imports are broken - fix before launching"

sys.path.insert(0, REPO)
os.chdir(REPO)

import torch
print("gpu   :", torch.cuda.get_device_name(0))
print("vram  :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
print("torch :", torch.__version__, "| cuda", torch.version.cuda)

cloned
torchdiffeq 0.2.5 | peft 0.19.1
torchao incompatible with peft - uninstalling (unused by this training path)
train.py imports OK
gpu   : NVIDIA A100-SXM4-80GB
vram  : 85.1 GB
torch : 2.6.0+cu124 | cuda 12.4


## 2. Drive, images, splits

In [ ]:
from google.colab import drive, userdata
import json

drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

DRIVE = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
IMGS, EXPECTED = "/content/imgs", 37063
TAR = f"{DRIVE}/imgs_subset.tar"

def n_files(d):
    return sum(len(f) for _, _, f in os.walk(d)) if os.path.isdir(d) else 0

if n_files(IMGS) < EXPECTED:
    os.makedirs(IMGS, exist_ok=True)
    subprocess.run(f'cp "{TAR}" /content/imgs_subset.tar && tar -xf /content/imgs_subset.tar -C {IMGS}',
                   shell=True, check=True)
    print("restored", n_files(IMGS), "/", EXPECTED)
    assert n_files(IMGS) >= EXPECTED, "restore incomplete"
else:
    print("images complete:", n_files(IMGS))

for s in ["train", "val", "screen"]:
    src = f"{DRIVE}/{s}.jsonl"
    if os.path.exists(src) and not os.path.exists(f"/content/{s}.jsonl"):
        subprocess.run(["cp", src, f"/content/{s}.jsonl"])
subprocess.run(["cp", f"{DRIVE}/screen_exclude_from_training.txt", "/content/"])
print("splits:", [s for s in ["train", "val", "screen"] if os.path.exists(f"/content/{s}.jsonl")])

Mounted at /content/drive
restored 37063 / 37063
splits: ['train', 'val', 'screen']


## 3. Preflight — VAE

`ft_lora.yml` points at the gated `black-forest-labs/FLUX.1-dev`. `OmniGen2/OmniGen2` ships its own `vae/`, which is what the inference pipeline loads. Compares config and weights; if identical the default is kept.

In [ ]:
from diffusers import AutoencoderKL

def load_vae(repo):
    try:
        return AutoencoderKL.from_pretrained(repo, subfolder="vae"), None
    except Exception as e:
        return None, str(e)[:300]

vae_og, err_og = load_vae("OmniGen2/OmniGen2")
print("OmniGen2/OmniGen2  vae:", "loaded" if vae_og else f"FAILED - {err_og}")

vae_fx, err_fx = load_vae("black-forest-labs/FLUX.1-dev")
print("FLUX.1-dev         vae:", "loaded" if vae_fx else f"FAILED (gated?) - {err_fx}")

if vae_og is not None and vae_fx is not None:
    cfg_same = dict(vae_og.config) == dict(vae_fx.config)
    a, b = vae_og.state_dict(), vae_fx.state_dict()
    keys_same = set(a) == set(b)
    w_same = keys_same and all(torch.equal(a[k], b[k]) for k in a)
    ca, cb = dict(vae_og.config), dict(vae_fx.config)
    diff = {k: (ca.get(k), cb.get(k)) for k in set(ca) | set(cb) if ca.get(k) != cb.get(k)}
    print("\nconfig fields that differ (OmniGen2 vs FLUX):")
    for k, (x, y) in sorted(diff.items()):
        print(f"    {k}: {x}  vs  {y}")
    for k in ("scaling_factor", "shift_factor"):
        print(f"  {k}: {ca.get(k)} vs {cb.get(k)}  -> "
              f"{'SAME' if ca.get(k) == cb.get(k) else 'DIFFERENT - latent scaling would not match inference'}")
    print(f"\nconfig identical : {cfg_same}")
    print(f"keys identical   : {keys_same}")
    print(f"weights identical: {w_same}")
    print("\nVERDICT:", "identical - keep the shipped default (FLUX.1-dev)" if w_same
          else "DIFFERENT - point the config at OmniGen2/OmniGen2 to match inference")
elif vae_og is not None:
    print("\nVERDICT: FLUX gated or unavailable - use OmniGen2/OmniGen2 (same VAE the pipeline loads)")

del vae_og, vae_fx
import gc; gc.collect(); torch.cuda.empty_cache()

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


config.json:   0%|          | 0.00/840 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

OmniGen2/OmniGen2  vae: loaded


config.json:   0%|          | 0.00/820 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

FLUX.1-dev         vae: loaded

config fields that differ (OmniGen2 vs FLUX):
    _diffusers_version: 0.33.1  vs  0.30.0.dev0
    _name_or_path: OmniGen2/OmniGen2  vs  black-forest-labs/FLUX.1-dev
  scaling_factor: 0.3611 vs 0.3611  -> SAME
  shift_factor: 0.1159 vs 0.1159  -> SAME

config identical : False
keys identical   : True
weights identical: True

VERDICT: identical - keep the shipped default (FLUX.1-dev)


## 4. Preflight — text encoder

The config loads stock `Qwen/Qwen2.5-VL-3B-Instruct`. OmniGen2 added an `<|img|>` token to its own copy, so the two may differ in vocab size and embedding shape. If they differ, training and inference would encode text with different weights.

In [ ]:
from transformers import AutoTokenizer

tok_q = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")
print("stock Qwen tokenizer vocab:", len(tok_q))

try:
    tok_o = AutoTokenizer.from_pretrained("OmniGen2/OmniGen2", subfolder="processor")
    print("OmniGen2 tokenizer vocab :", len(tok_o))
    print("vocab sizes match        :", len(tok_q) == len(tok_o))
    extra = set(tok_o.get_vocab()) - set(tok_q.get_vocab())
    print("tokens only in OmniGen2  :", sorted(extra)[:10] if extra else "none")
except Exception as e:
    print("OmniGen2 tokenizer load failed:", str(e)[:300])

from huggingface_hub import list_repo_files
files = list_repo_files("OmniGen2/OmniGen2")
print("\nOmniGen2 repo subfolders:", sorted({f.split('/')[0] for f in files if '/' in f}))
# the 4 extra tokens appear in 0 of 39,893 training instructions and added tokens are appended,
# so shared token IDs are unchanged; this checks the shared weights are unchanged too
from transformers import Qwen2_5_VLModel as TextEncoder

te_q = TextEncoder.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct", torch_dtype=torch.bfloat16)
te_o = TextEncoder.from_pretrained("OmniGen2/OmniGen2", subfolder="mllm", torch_dtype=torch.bfloat16)
sq, so = te_q.state_dict(), te_o.state_dict()

shared = sorted(set(sq) & set(so))
print(f"\nshared tensors: {len(shared)} | only in Qwen: {len(set(sq)-set(so))} | only in OmniGen2: {len(set(so)-set(sq))}")

mismatch = []
for k in shared:
    a, b = sq[k], so[k]
    if a.shape != b.shape:
        n = min(a.shape[0], b.shape[0])          # embedding matrix: compare the shared vocab rows
        if not torch.equal(a[:n], b[:n]):
            mismatch.append((k, "shared rows differ"))
    elif not torch.equal(a, b):
        mismatch.append((k, "values differ"))

print("tensors that differ:", [k for k, _ in mismatch] or "none")

confined = None
ek = [k for k in shared if k.endswith("embed_tokens.weight")]
if ek:
    k = ek[0]
    a, b = sq[k].float(), so[k].float()
    n = min(a.shape[0], b.shape[0])
    rows = (a[:n] != b[:n]).any(dim=1).nonzero().flatten().tolist()
    new_ids = sorted(tok_o.convert_tokens_to_ids(t)
                     for t in ["<|endofimg|>", "<|endofmeta|>", "<|img|>", "<|meta|>"])
    print(f"\n{k}: shapes {tuple(sq[k].shape)} vs {tuple(so[k].shape)}")
    print(f"  rows that differ  : {rows if len(rows) <= 12 else str(rows[:12]) + f' ...({len(rows)} total)'}")
    print(f"  ids of the 4 new  : {new_ids}")
    confined = set(rows) <= set(new_ids)
    print(f"  confined to the 4 new tokens: {confined}")

    ids_used = set()
    for j, line in enumerate(open("/content/train.jsonl", encoding="utf-8")):
        if j >= 500:
            break
        ids_used |= set(tok_q(json.loads(line)["instruction"])["input_ids"])
    print(f"  differing rows that our instructions actually use: {sorted(set(rows) & ids_used) or 'none'}")

others = [k for k, _ in mismatch if not k.endswith("embed_tokens.weight")]
ok = not others and confined is not False
print("\nVERDICT:", "safe - only the 4 unused special-token rows differ; keep the default (stock Qwen)" if ok
      else "DIFFERENT beyond the new tokens - point the text encoder at the OmniGen2 mllm copy")

del te_q, te_o, sq, so
import gc; gc.collect()

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

stock Qwen tokenizer vocab: 151665


...

mllm/model-00004-of-00004.safetensors:   0%|          | 0.00/180M [00:00<?, ?B/s]

mllm/model-00002-of-00004.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]


shared tensors: 434 | only in Qwen: 0 | only in OmniGen2: 0
tensors that differ: ['embed_tokens.weight']

embed_tokens.weight: shapes (151936, 2048) vs (151936, 2048)
  rows that differ  : [151665]
  ids of the 4 new  : [151665, 151666, 151667, 151668]
  confined to the 4 new tokens: True
  differing rows that our instructions actually use: none

VERDICT: safe - only the 4 unused special-token rows differ; keep the default (stock Qwen)


73

## 5. Timing data

300 training rows in OmniGen2's format: `task_type: "ic"`, absolute paths, screen targets excluded. Enough rows that the loader never repeats within 12 steps x global batch 16 = 192 samples.

In [ ]:
DATA = "/content/lora_data"
os.makedirs(f"{DATA}/jsonls", exist_ok=True)

excl = set(open("/content/screen_exclude_from_training.txt").read().split())
train = [json.loads(l) for l in open("/content/train.jsonl", encoding="utf-8")]
kept = [r for r in train if r["output_image"] not in excl]
print(f"train {len(train)} rows -> {len(kept)} after removing {len(excl)} screen targets")
assert len(kept) == 39591, f"expected 39591, got {len(kept)}"

def convert(r):
    return {"task_type": "ic",
            "instruction": r["instruction"],
            "input_images": [os.path.join(IMGS, p) for p in r["input_images"]],
            "output_image": os.path.join(IMGS, r["output_image"])}

rows = [convert(r) for r in kept[:300]]
missing = [p for r in rows for p in r["input_images"] + [r["output_image"]] if not os.path.exists(p)]
assert not missing, f"{len(missing)} missing images, first: {missing[:3]}"

with open(f"{DATA}/jsonls/timing.jsonl", "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

open(f"{DATA}/ic.yml", "w").write(
    'data:\n  - \n    path: "%s/jsonls/timing.jsonl"\n    type: "ic"\n    ratio: !!float 1\n' % DATA)
open(f"{DATA}/mix.yml", "w").write(
    "data:\n  - \n    path: '%s/ic.yml'\n    type: 'ic'\n    ratio: !!float 1\n" % DATA)

print(f"wrote {len(rows)} rows, all images present")
print(json.dumps(rows[0], indent=2)[:420])

train 39893 rows -> 39591 after removing 62 screen targets
wrote 300 rows, all images present
{
  "task_type": "ic",
  "instruction": "Three laptops are displayed side by side on a sleek glass table in a modern office. The laptop on the left is a vibrant red model, featuring a glossy finish and a backlit keyboard, perfect for catching attention during presentations. The middle laptop is a minimalist silver design, boasting a slim profile and a high-resolution display, ideal for professional tasks and video ed


## 6. Timing config

`ft_lora.yml` with four changes: three resolution keys to 512, and `gradient_accumulation_steps: 8` so `global_batch_size: 16` still holds on one GPU. Steps cut to 12 and checkpointing pushed out of range.

In [ ]:
from omegaconf import OmegaConf

VAE_REPO = "black-forest-labs/FLUX.1-dev"   # set to OmniGen2/OmniGen2 if section 3 says DIFFERENT

c = OmegaConf.load(f"{REPO}/options/ft_lora.yml")

c.name = "timing_test"
c.data.data_path = f"{DATA}/mix.yml"
c.model.pretrained_vae_model_name_or_path = VAE_REPO

c.data.max_output_pixels = 262144          # 512 * 512
c.data.max_input_pixels = [262144] * 4     # refs, matching load_inputs(max_side=512)
c.data.max_side_length = 512

c.train.gradient_accumulation_steps = 8    # keeps global_batch_size 16 on one GPU
c.train.max_train_steps = 12
c.logger.checkpointing_steps = 10 ** 9     # no checkpoint during timing
c.logger.log_with = ["tensorboard"]        # wandb would block on login
c.val.train_visualization_steps = 10 ** 9  # no image dumps during timing

OmegaConf.save(c, f"{REPO}/options/timing_test.yml")

print("changed vs ft_lora.yml:")
base = OmegaConf.load(f"{REPO}/options/ft_lora.yml")
for path in ["data.max_output_pixels", "data.max_input_pixels", "data.max_side_length",
             "train.gradient_accumulation_steps", "train.max_train_steps",
             "logger.checkpointing_steps", "logger.log_with", "val.train_visualization_steps",
             "model.pretrained_vae_model_name_or_path"]:
    print(f"  {path:42s} {OmegaConf.select(base, path)}  ->  {OmegaConf.select(c, path)}")
print("\nunchanged: lr 8e-7, warmup 500, global_batch_size 16, batch_size 2, lora_rank 8,")
print("           lora_dropout 0, ref_img_dropout_prob 0.5, prompt_dropout_prob 1e-4,")
print("           maximum_text_tokens 888, gradient_checkpointing True, bf16, seed 2233")

changed vs ft_lora.yml:
  data.max_output_pixels                     1048576  ->  262144
  data.max_input_pixels                      [1048576, 1048576, 589824, 262144]  ->  [262144, 262144, 262144, 262144]
  data.max_side_length                       2048  ->  512
  train.gradient_accumulation_steps          1  ->  8
  train.max_train_steps                      4000  ->  12
  logger.checkpointing_steps                 1000  ->  1000000000
  logger.log_with                            ['wandb', 'tensorboard']  ->  ['tensorboard']
  val.train_visualization_steps              100  ->  1000000000
  model.pretrained_vae_model_name_or_path    black-forest-labs/FLUX.1-dev  ->  black-forest-labs/FLUX.1-dev

unchanged: lr 8e-7, warmup 500, global_batch_size 16, batch_size 2, lora_rank 8,
           lora_dropout 0, ref_img_dropout_prob 0.5, prompt_dropout_prob 1e-4,
           maximum_text_tokens 888, gradient_checkpointing True, bf16, seed 2233


## 7. Timing patch

Prints wall time and peak VRAM after each optimizer step. Idempotent; `git checkout train.py` reverts.

In [ ]:
p = f"{REPO}/train.py"
src = open(p, encoding="utf-8").read()

ANCHOR = "                global_step += 1\n"
PATCH = '''                if accelerator.is_main_process:
                    import time as _t
                    _g = globals().setdefault("_TIMING", {"prev": None, "log": []})
                    _now = _t.time()
                    if _g["prev"] is not None:
                        _dt = _now - _g["prev"]
                        _g["log"].append(_dt)
                        print("[timing] step %d %.3fs peak_vram %.1fGB" % (
                            global_step, _dt, torch.cuda.max_memory_allocated() / 1e9), flush=True)
                    _g["prev"] = _now
'''

if "[timing]" in src:
    print("already patched")
else:
    assert src.count(ANCHOR) == 1, f"anchor found {src.count(ANCHOR)} times; inspect train.py"
    open(p, "w", encoding="utf-8").write(src.replace(ANCHOR, ANCHOR + PATCH))
    import ast; ast.parse(open(p, encoding="utf-8").read())
    print("patched and parses")

patched and parses


## 8. Run

Same launch flags as `scripts/train/ft_lora.sh` with `num_processes=1`.

In [ ]:
import time

cmd = [
    "accelerate", "launch",
    "--num_processes=1", "--num_machines=1", "--machine_rank=0", "--main_process_port=29500",
    "--use_fsdp",
    "--fsdp_offload_params", "false",
    "--fsdp_sharding_strategy", "HYBRID_SHARD_ZERO2",
    "--fsdp_auto_wrap_policy", "TRANSFORMER_BASED_WRAP",
    "--fsdp_transformer_layer_cls_to_wrap", "OmniGen2TransformerBlock",
    "--fsdp_state_dict_type", "FULL_STATE_DICT",
    "--fsdp_forward_prefetch", "false",
    "--fsdp_use_orig_params", "True",
    "--fsdp_cpu_ram_efficient_loading", "false",
    "--fsdp_sync_module_states", "True",
    "train.py", "--config", "options/timing_test.yml",
]
print(" ".join(cmd), "\n")

t0 = time.time()
log = []
proc = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1, env={**os.environ, "WANDB_DISABLED": "true"})
for line in proc.stdout:
    log.append(line)
    if "[timing]" in line or "Error" in line or "error" in line or "Traceback" in line:
        print(line.rstrip())
proc.wait()
print(f"\nexit {proc.returncode} | wall {time.time() - t0:.0f}s")
open("/content/timing_log.txt", "w").write("".join(log))
if proc.returncode != 0:
    print("\n--- last 40 lines ---")
    print("".join(log[-40:]))

accelerate launch --num_processes=1 --num_machines=1 --machine_rank=0 --main_process_port=29500 --use_fsdp --fsdp_offload_params false --fsdp_sharding_strategy HYBRID_SHARD_ZERO2 --fsdp_auto_wrap_policy TRANSFORMER_BASED_WRAP --fsdp_transformer_layer_cls_to_wrap OmniGen2TransformerBlock --fsdp_state_dict_type FULL_STATE_DICT --fsdp_forward_prefetch false --fsdp_use_orig_params True --fsdp_cpu_ram_efficient_loading false --fsdp_sync_module_states True train.py --config options/timing_test.yml 

[timing] step 2 16.717s peak_vram 44.1GB
[timing] step 3 15.350s peak_vram 44.8GB
[timing] step 4 13.644s peak_vram 44.8GB
[timing] step 5 15.482s peak_vram 44.8GB
[timing] step 6 14.698s peak_vram 45.0GB
[timing] step 7 13.225s peak_vram 45.0GB
[timing] step 8 14.655s peak_vram 45.0GB
[timing] step 9 12.758s peak_vram 45.0GB
[timing] step 10 15.686s peak_vram 45.5GB
[timing] step 11 14.852s peak_vram 45.5GB
[timing] step 12 14.007s peak_vram 45.5GB

exit 0 | wall 291s


## 9. Result and extrapolation

In [ ]:
import re, statistics as st

txt = open("/content/timing_log.txt").read()
steps = [(int(a), float(b), float(c)) for a, b, c in
         re.findall(r"\[timing\] step (\d+) ([\d.]+)s peak_vram ([\d.]+)GB", txt)]

assert steps, "no timing lines; check section 8 output"
for s, dt, vram in steps:
    print(f"  step {s:>3}  {dt:6.2f}s  peak {vram:.1f} GB")

warm = [dt for s, dt, _ in steps if s > steps[0][0] + 1]      # drop first two: allocator warmup
assert warm, "not enough steps to discard warmup"
s_opt = st.median(warm)
s_micro = s_opt / 8
peak = max(v for _, _, v in steps)

print(f"\nmedian optimizer step : {s_opt:.2f}s   (n={len(warm)}, warmup discarded)")
print(f"per micro-step        : {s_micro:.2f}s   (8 per optimizer step)")
print(f"peak VRAM             : {peak:.1f} GB")

UNITS_PER_H, EUR_PER_UNIT, HAVE = 11.77, 50.28 / 500, 174.91
h_rank = 4000 * s_opt / 3600

print(f"\n{'':26s} {'hours':>8s} {'units':>8s} {'EUR':>8s}")
print("-" * 54)
for lab, h in [("training, 1 rank", h_rank),
               ("training, 3 ranks", 3 * h_rank),
               ("screen evals (6 x 302)", 6 * 302 * 23.2 / 3600),
               ("test finalist (488)", 488 * 23.2 / 3600),
               ("val loss (negligible)", 0.4)]:
    print(f"{lab:26s} {h:8.1f} {h*UNITS_PER_H:8.0f} {h*UNITS_PER_H*EUR_PER_UNIT:8.2f}")

total_h = 3*h_rank + 6*302*23.2/3600 + 488*23.2/3600 + 0.4
total_u = total_h * UNITS_PER_H
print("-" * 54)
print(f"{'LoRA ARM TOTAL':26s} {total_h:8.1f} {total_u:8.0f} {total_u*EUR_PER_UNIT:8.2f}")
print(f"\nhave {HAVE:.2f} units -> shortfall {max(0, total_u - HAVE):.0f} units "
      f"({max(0, total_u - HAVE) * EUR_PER_UNIT:.2f} EUR)")
print(f"with the 500-pack: {HAVE + 500:.0f} units, "
      f"{'sufficient' if HAVE + 500 > total_u else 'STILL SHORT'}, "
      f"{HAVE + 500 - total_u:.0f} units spare")

if peak > 38:
    print("\nNOTE: peak VRAM above 38 GB - a 40 GB A100 would be marginal.")
if s_opt > 8:
    print("\nNOTE: slower than the 5-8s estimate; re-check the plan before committing units.")

  step   2   16.72s  peak 44.1 GB
  step   3   15.35s  peak 44.8 GB
  step   4   13.64s  peak 44.8 GB
  step   5   15.48s  peak 44.8 GB
  step   6   14.70s  peak 45.0 GB
  step   7   13.22s  peak 45.0 GB
  step   8   14.65s  peak 45.0 GB
  step   9   12.76s  peak 45.0 GB
  step  10   15.69s  peak 45.5 GB
  step  11   14.85s  peak 45.5 GB
  step  12   14.01s  peak 45.5 GB

median optimizer step : 14.65s   (n=9, warmup discarded)
per micro-step        : 1.83s   (8 per optimizer step)
peak VRAM             : 45.5 GB

                              hours    units      EUR
------------------------------------------------------
training, 1 rank               16.3      192    19.27
training, 3 ranks              48.9      575    57.82
screen evals (6 x 302)         11.7      137    13.82
test finalist (488)             3.1       37     3.72
val loss (negligible)           0.4        5     0.47
------------------------------------------------------
LoRA ARM TOTAL                 64.1      754  

## 10. Revert the patch

In [ ]:
print(subprocess.run(["git", "checkout", "--", "train.py"], cwd=REPO,
                     capture_output=True, text=True).stderr or "train.py reverted")
print("timing_test.yml kept at options/timing_test.yml")

train.py reverted
timing_test.yml kept at options/timing_test.yml
